# Analyze
### No Claude Used!
**Author:** Elisa Warner

In [205]:
import pandas as pd
import os
import numpy as np

In [279]:
addresses = [x for x in os.listdir("library") if not x.startswith(".")]

In [318]:
class CalcFeatures(object):
    def __init__(self, address):
        self.address = address
        
    def totalTrans(self, df):
        return df.shape[0]
    
    def outCount(self, df):
        return df[df['from'] == address.lower()].shape[0]
    
    def inCount(self, df):
        return df[df['to'] == address.lower()].shape[0]
    
    def outSum(self, df):
        return df[df['from'] == address.lower()]['value'].sum()
    
    def outAmtMean(self, df):
        return df[df['from'] == address.lower()]['value'].mean()
    
    def outAmtMin(self, df):
        return df[df['from'] == address.lower()]['value'].min()
    
    def outAmtMax(self, df):
        return df[df['from'] == address.lower()]['value'].max()

    def inSum(self, df):
        return  df[df['to'] == address.lower()]['value'].sum()

    def inAmtMean(self, df):
        return df[df['to'] == address.lower()]['value'].mean()

    def inAmtMin(self, df):
        return df[df['to'] == address.lower()]['value'].min()

    def inAmtMax(self, df):
        return df[df['to'] == address.lower()]['value'].max()

    def tokenCount(self, df):
        return df.groupby("tokenName").count()['blockNumber'].to_dict()

    def tokenSum(self, df):
        return df.groupby("tokenName")['value'].sum().to_dict()

    def tokenMin(self, df):
        return df.groupby("tokenName")['value'].min().to_dict()

    def tokenMax(self, df):
        return df.groupby("tokenName")['value'].max().to_dict()

    def tokenMean(self, df):
        return df.groupby("tokenName")['value'].mean().to_dict()

    def ageAcct(self, df):
        try:
            return (df.timeStamp.max() - df.timeStamp.min()).total_seconds()
        except:
            return 0

    def dateOfFirstTxn(self, df):
        return df.timeStamp.min()

    def addressName(self):
        return self.address
    
    def calc_features(self, feature_list, df):
        output = []

        for f in feature_list:
            if f == "address_name":
                output.append(self.addressName())
            if f == "first_txn_dt":
                output.append(self.dateOfFirstTxn(df))
            if f == "total_trans":
                output.append(self.totalTrans(df))
            if f == "out_c":
                output.append(self.outCount(df))
            if f == "in_c":
                output.append(self.inCount(df))
            if f == "out_s":
                output.append(self.outSum(df))
            if f == "out_amt_mean":
                output.append(self.outAmtMean(df))
            if f == "out_amt_min":
                output.append(self.outAmtMin(df))
            if f == "out_amt_max":
                output.append(self.outAmtMax(df))
            if f == "in_s":
                output.append(self.inSum(df))
            if f == "in_amt_mean":
                output.append(self.inAmtMean(df))
            if f == "in_amt_min":
                output.append(self.inAmtMin(df))
            if f == "in_amt_max":
                output.append(self.inAmtMax(df))
            if f == "age_acct_sec":
                output.append(self.ageAcct(df))
            if f == "token_count":
                output_dict = self.tokenCount(df)
                other = np.sum([output_dict[x] for x in output_dict if x != 'Tether USD' and x != 'USD Coin'])
                output = output + [output_dict.get('Tether USD', 0), output_dict.get('USD Coin', 0), other]
            if f == "token_sum":
                output_dict = self.tokenSum(df)
                other = np.sum([output_dict[x] for x in output_dict if x != 'Tether USD' and x != 'USD Coin'])
                output = output + [output_dict.get('Tether USD', 0), output_dict.get('USD Coin', 0), other]
            if f == "token_min":
                output_dict = self.tokenMin(df)
                other = np.sum([output_dict[x] for x in output_dict if x != 'Tether USD' and x != 'USD Coin'])
                output = output + [output_dict.get('Tether USD', 0), output_dict.get('USD Coin', 0), other]
            if f == "token_max":
                output_dict = self.tokenMax(df)
                other = np.sum([output_dict[x] for x in output_dict if x != 'Tether USD' and x != 'USD Coin'])
                output = output + [output_dict.get('Tether USD', 0), output_dict.get('USD Coin', 0), other]
            if f == "token_mean":
                output_dict = self.tokenMean(df)
                other = np.sum([output_dict[x] for x in output_dict if x != 'Tether USD' and x != 'USD Coin'])
                output = output + [output_dict.get('Tether USD', 0), output_dict.get('USD Coin', 0), other]

        return output

In [319]:
aggregated_feature_df = pd.DataFrame()

features = ['first_txn_dt',
             'total_trans',
             'out_c',
             'in_c',
             'out_s',
             'out_amt_mean',
             'out_amt_min',
             'out_amt_max',
             'in_s',
             'in_amt_mean',
             'in_amt_min',
             'in_amt_max',
             'age_acct_sec']

features_erc20 = ['address_name',
                  'token_count',
                  'token_sum',
                  'token_min',
                  'token_max',
                  'token_mean']

for address in addresses:
    temp_row = pd.DataFrame()
    
    for filetype in ['normal', 'erc-20-tokens', 'internal']:
        calculator = CalcFeatures(address)
        print(calculator.address, filetype)
        
        try:
            df = pd.read_csv("library/%s/%s_transactions.csv" % (address, filetype))

            df.timeStamp = pd.to_datetime(df.timeStamp, unit="s")
            
            # conversion of value to match etherscan
            if filetype == "normal" or filetype == "internal":
                df['value'] = df['value'] / (1000000000000000000)
            if filetype == "erc-20-tokens":
                df['value'] = df['value'] / 1000000
            
        except:
            df = pd.DataFrame(columns = ['timeStamp','value', 'tokenName', 'from', 'to', 'blockNumber'])

        feature_names = ['%s_first_txn_dt' % filetype,
             '%s_total_transactions' % filetype,
             '%s_total_count_out' % filetype,
             '%s_total_count_in' % filetype,
             '%s_total_sum_out' % filetype,
             '%s_mean_amt_out' % filetype,
             '%s_min_amt_out' % filetype,
             '%s_max_amt_out' % filetype,
             '%s_total_sum_in' % filetype,
             '%s_mean_amt_in' % filetype,
             '%s_min_amt_in' % filetype,
             '%s_max_amt_in' % filetype,
             '%s_age_acct_seconds' % filetype]

        feature_names_erc20 = ['address',
                   '%s_count_usdt' % filetype,
                   '%s_count_usdc' % filetype,
                   '%s_count_other' % filetype,
                   '%s_sum_usdt' % filetype,
                   '%s_sum_usdc' % filetype,
                   '%s_sum_other' % filetype,
                   '%s_min_usdt' % filetype,
                   '%s_min_usdc' % filetype,
                   '%s_min_other' % filetype,
                   '%s_max_usdt' % filetype,
                   '%s_max_usdc' % filetype,
                   '%s_max_other' % filetype,
                   '%s_mean_usdt' % filetype,
                   '%s_mean_usdc' % filetype,
                   '%s_mean_other' % filetype]

        if filetype == "erc-20-tokens":
            featureList = features + features_erc20
            featureNames = feature_names + feature_names_erc20
        else:
            featureList = features
            featureNames = feature_names
                               
        output_features = calculator.calc_features(featureList, df)
        
        temp = pd.DataFrame(output_features)
        temp = temp.T
        temp.columns = featureNames

        temp_row = pd.concat((temp_row, temp), axis=1)
    
    aggregated_feature_df = pd.concat((aggregated_feature_df, temp_row), axis=0)

0xB5359336b305C8db8a89524f115528cF7aD89eA2 normal
0xB5359336b305C8db8a89524f115528cF7aD89eA2 erc-20-tokens
0xB5359336b305C8db8a89524f115528cF7aD89eA2 internal
0x7174d846b27fd468853303895aBb8dbE95E44808 normal
0x7174d846b27fd468853303895aBb8dbE95E44808 erc-20-tokens
0x7174d846b27fd468853303895aBb8dbE95E44808 internal
0x0A76D0C88683Dc3AcFEb4DEdF47064a9B44e8699 normal
0x0A76D0C88683Dc3AcFEb4DEdF47064a9B44e8699 erc-20-tokens
0x0A76D0C88683Dc3AcFEb4DEdF47064a9B44e8699 internal
0x641C0882b0De34308db18310DC080B736A673bA1 normal
0x641C0882b0De34308db18310DC080B736A673bA1 erc-20-tokens
0x641C0882b0De34308db18310DC080B736A673bA1 internal
0x30a5cbe88A7fE348fc2902e98C38bba36fA614D7 normal
0x30a5cbe88A7fE348fc2902e98C38bba36fA614D7 erc-20-tokens
0x30a5cbe88A7fE348fc2902e98C38bba36fA614D7 internal
0x02f03E4AEfc35dF901AC734DEC9e028A708A4Ad9 normal
0x02f03E4AEfc35dF901AC734DEC9e028A708A4Ad9 erc-20-tokens
0x02f03E4AEfc35dF901AC734DEC9e028A708A4Ad9 internal
0xd4AA54A29e359fBe037d8490bF37F2A23256A866 nor

In [320]:
allfeatures = list(aggregated_feature_df)
allfeatures.pop(allfeatures.index('address'))

aggregated_feature_df = aggregated_feature_df[['address'] + allfeatures]

In [321]:
aggregated_feature_df 

,address,normal_first_txn_dt,normal_total_transactions,normal_total_count_out,normal_total_count_in,normal_total_sum_out,normal_mean_amt_out,normal_min_amt_out,normal_max_amt_out,normal_total_sum_in,...,internal_total_count_in,internal_total_sum_out,internal_mean_amt_out,internal_min_amt_out,internal_max_amt_out,internal_total_sum_in,internal_mean_amt_in,internal_min_amt_in,internal_max_amt_in,internal_age_acct_seconds
0,0xB5359336b305C8db8a89524f115528cF7aD89eA2,2026-04-23 18:07:23,50,42,8,2.194868,0.052259,0.0,0.617434,1.275974,...,4,0.0,NaN,NaN,NaN,0.95125,0.237812,0.010139,0.465486,1452.0
0,0x7174d846b27fd468853303895aBb8dbE95E44808,2026-01-31 02:30:35,175,115,61,5.273957,0.04586,0.0,1.1396,5.001713,...,22,0.0,NaN,NaN,NaN,2.8447,0.129305,0.0,0.5,1288536.0
0,0x0A76D0C88683Dc3AcFEb4DEdF47064a9B44e8699,2024-08-26 17:22:23,56,44,12,7.784633,0.176923,0.0,1.4797,7.849575,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0
0,0x641C0882b0De34308db18310DC080B736A673bA1,2026-03-07 01:31:47,8,4,4,0.0,0.0,0.0,0.0,0.00023,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0
0,0x30a5cbe88A7fE348fc2902e98C38bba36fA614D7,2026-01-08 19:53:11,63,30,33,0.008655,0.000289,0.0,0.008655,0.017723,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0
0,0x02f03E4AEfc35dF901AC734DEC9e028A708A4Ad9,2025-01-11 12:26:35,146,94,52,175.135698,1.863146,0.0,21.5086,175.226746,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0
0,0xd4AA54A29e359fBe037d8490bF37F2A23256A866,2026-02-11 17:41:23,46,10,36,96.8296,9.68296,0.1074,22.2,96.616617,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0
0,0x6f92dE21f6E34b1bcAAD7A33dE5B3a5e370eEf11,2026-03-26 03:00:11,4,4,0,0.092673,0.023168,0.022011,0.024326,0.0,...,4,0.0,NaN,NaN,NaN,0.092679,0.02317,0.022011,0.024328,682368.0
0,0xddbd2b932c763ba5b1b7ae3b362eac3e8d40121a,2015-08-07 11:45:53,183,56,127,208515.624,3723.493286,0.0,10600.0,169109.581167,...,12,0,NaN,NaN,NaN,81214.0,6767.833333,7.0,10600.0,34304.0
0,0x6a7b8d6032640e37c1db7b44378fa7bc063002f9,2025-03-09 09:21:59,310,126,184,204.974428,1.626781,0.0,9.739,216.365041,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0


In [322]:
aggregated_feature_df.to_csv("aggregated_features_addresses.csv", index=False)